# Create Einstein Foundation Berlin Awards

Creates Einstein Stiftung Berlin (Einstein Foundation Berlin) awards from the
foundation's own "Fellows & Projects" directory at
https://www.einsteinfoundation.de/en/fellows-projects (25 list pages of
tiles + one detail page per tile). **290 awards: 184 people (Einstein
Visiting Fellows, Professorships, Strategic Professorships, BIH and
BUA/Oxford Visiting Fellows, International Postdoctoral / Junior / Research
Fellows, Postdoctoral Grants, Starting Researchers) and 106 projects or
structures (Einstein Research Projects, Berlin/HUJI projects, Circles,
Research Units, Centers, Doctoral Programs)** (local run 2026-10-01).

**Prerequisites:**
- Run `scripts/local/einstein_foundation_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/einstein_foundation/einstein_foundation_projects.parquet`
  (runbook §1.4 shrink guard).

**Field notes:**
- `display_name`: projects/structures use the detail page heading (tile names
  are truncated); people use "{name} — {programme}" (HHMI pattern, §2.3.1),
  since person pages carry no project title.
- `funder_scheme`: the programme, from the detail URL path. `funding_type`:
  `fellowship` for the person programmes, `training` for the Einstein
  Foundation Doctoral Program, `research` otherwise. Scope filter: none;
  every entry is research, fellowship or doctoral-training funding.
- Dates: people from the "Short Vita" line naming the Einstein programme
  ("2010 - 2014 | Einstein Visiting Fellow at ..."; "Since 2020 | Einstein
  Professor ..." = open-ended); projects from "Funding period:"/"Duration:".
  Month-level periods give exact dates; year-only ones use 1 Jan / 31 Dec.
  ~88% have a start year; Circles and most Centers publish no period.
- `lead_investigator`: people = the funded person; projects, Circles and
  Research Units = the first titled person under "Contact"/"Applicant"
  (honorifics stripped, canonical `split_name`). Centers and Doctoral
  Programs: NULL (their contact is an office/coordinator, not a PI).
  Affiliation = the Berlin host institution from the directory tile
  (normalised spelling, country DE); for Visiting Fellows that is the host,
  not their home university.
- `amount`/`currency`: NULL. The site publishes no per-award amounts (only
  occasional programme totals in prose).

**`funder_award_id` (§2.1.1):** citing works quote the foundation's internal
numbers (`EVF-2020-571`, `A-2012_114`, `IPF-2012-148`, ~520 priority-0
shells), which the site never shows. The ingest uses the synthetic, stable
key `ESB-{programme path}/{item slug}` and collapses onto **0** existing
citation stubs (checked in the last verification cell).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320323688` (exactly one row in `openalex.funders.funders`; no
  twin by name or ROR. Not to be confused with the Einstein Centers that are
  their own funders, e.g. F4320336372 ECRT, F2298238354 ECN.)
- display_name: Einstein Stiftung Berlin
- ror_id / doi: from `openalex.funders.funders` (ror 03s0fv852, doi 10.13039/501100006188)

**Priority:** `586` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.einstein_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/einstein_foundation/einstein_foundation_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT funder_award_id) as distinct_ids
FROM openalex.awards.einstein_foundation_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.einstein_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.einstein_foundation_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320323688 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323688;

## Step 2: Create Einstein Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.einstein_foundation_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323688  -- Einstein Stiftung Berlin
),

g AS (
    SELECT
        r.*,
        TRIM(r.funder_award_id) AS award_key,
        COALESCE(TRY_TO_DATE(r.start_date, 'yyyy-MM-dd'),
                 CASE WHEN TRY_CAST(r.start_year AS INT) IS NOT NULL THEN make_date(TRY_CAST(r.start_year AS INT), 1, 1) END) AS start_dt,
        COALESCE(TRY_TO_DATE(r.end_date, 'yyyy-MM-dd'),
                 CASE WHEN TRY_CAST(r.end_year AS INT) IS NOT NULL THEN make_date(TRY_CAST(r.end_year AS INT), 12, 31) END) AS end_dt,
        CASE WHEN NULLIF(TRIM(r.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.university), ''),
                'country', CASE WHEN NULLIF(TRIM(r.university), '') IS NOT NULL THEN 'DE' END,
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.einstein_foundation_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000 as id,
    NULLIF(TRIM(g.award_title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    g.award_key as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'einstein_foundation_directory' as provenance,
    g.start_dt as start_date,
    g.end_dt as end_date,
    YEAR(g.start_dt) as start_year,
    YEAR(g.end_dt) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.award_key)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE g.award_key IS NOT NULL AND g.award_key != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'einstein_foundation_directory' AND priority = 586;

-- Insert into openalex_awards_raw with priority.
-- Priority 586: direct-from-funder ingest of the Einstein Foundation Berlin
-- Fellows & Projects directory. Higher wins under the 2026-06-20 DESC dedup.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    586 as priority
FROM openalex.awards.einstein_foundation_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.einstein_foundation_awards;

In [ ]:
%sql
-- §2.1.1 shape check: every funder_award_id is ESB-{programme path}/{slug}.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^ESB-einstein-[a-z0-9/-]+$' THEN 1 ELSE 0 END) as ok_shape,
    SUM(CASE WHEN funder_award_id NOT RLIKE '^ESB-einstein-[a-z0-9/-]+$' THEN 1 ELSE 0 END) as bad_shape
FROM openalex.awards.einstein_foundation_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.einstein_foundation_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, COUNT(start_year) as with_year,
       COUNT(lead_investigator) as with_lead
FROM openalex.awards.einstein_foundation_awards
GROUP BY funder_scheme, funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.einstein_foundation_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.einstein_foundation_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.einstein_foundation_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (expect 100% title, ~88% start year, ~91% lead, no amounts).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.einstein_foundation_awards;

In [ ]:
%sql
-- Overlap with existing citation stubs for this funder (expect 0: synthetic key).
SELECT c.provenance, COUNT(*) as stubs, COUNT(t.id) as collapsed_onto_esb_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.einstein_foundation_awards t ON t.id = c.id
WHERE c.funder_id = 4320323688 AND c.provenance != 'einstein_foundation_directory'
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'einstein_foundation_directory'
GROUP BY provenance, priority;